# SASRec on MovieLens1M with Chronological Split

This Colab notebook is designed to run SASRec on the **same MovieLens1M data source used by the LightGCN notebook**, but with a **chronological leave-last-two-out split** so SASRec can preserve its sequential recommendation setting.

Main setup:

- Dataset: MovieLens1M `ratings.dat`
- Split: per-user chronological split
  - train = `seq[:-2]`
  - validation target = `seq[-2]`
  - test target = `seq[-1]`
- Model: SASRec
- Evaluation: Recall@10, NDCG@10, Recall@20, NDCG@20

Before running, select **Runtime → Change runtime type → GPU**.

## 1. Check GPU

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Go to Runtime → Change runtime type → GPU.')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Download MovieLens1M

This uses the official GroupLens MovieLens1M file, equivalent to the raw `ratings.dat` used in the LightGCN notebook.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/data/ml-1m')
RATINGS_PATH = DATA_DIR / 'ml-1m' / 'ratings.dat'

if not RATINGS_PATH.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/data/ml-1m.zip
    !unzip -q /content/data/ml-1m.zip -d /content/data/ml-1m

print('ratings.dat exists:', RATINGS_PATH.exists())
print('ratings.dat path:', RATINGS_PATH)
!ls -lh /content/data/ml-1m/ml-1m | head

ratings.dat exists: True
ratings.dat path: /content/data/ml-1m/ml-1m/ratings.dat
total 24M
-rw-r----- 1 root root 168K Mar 26  2003 movies.dat
-rw-r----- 1 root root  24M Feb 28  2003 ratings.dat
-rw-r----- 1 root root 5.5K Jan 29  2016 README
-rw-r----- 1 root root 132K Feb 28  2003 users.dat


## 3. Imports and experiment config

The default config follows our current best SASRec direction, but you can lower epochs for a smoke test.

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ===== Main experiment config =====
CONFIG = {
    'ratings_path': str(RATINGS_PATH),
    'min_user_interactions': 5,
    'max_len': 20,
    'hidden_dim': 128,
    'num_heads': 4,
    'num_layers': 2,
    'dropout': 0.3,
    'batch_size': 512,
    'epochs': 30,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'topks': [10, 20],
    'save_path': '/content/sasrec_movielens1m_best.pth',
}

CONFIG

Device: cuda


{'ratings_path': '/content/data/ml-1m/ml-1m/ratings.dat',
 'min_user_interactions': 5,
 'max_len': 20,
 'hidden_dim': 128,
 'num_heads': 4,
 'num_layers': 2,
 'dropout': 0.3,
 'batch_size': 512,
 'epochs': 30,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'topks': [10, 20],
 'save_path': '/content/sasrec_movielens1m_best.pth'}

## 4. Data loading and chronological split

In [ ]:
def load_ratings(path):
    """
    Load MovieLens ratings.

    Supports:
    - MovieLens CSV with columns userId, movieId, rating, timestamp
    - MovieLens1M ratings.dat with format userId::movieId::rating::timestamp
    """
    path = str(path)
    if path.endswith('.dat'):
        df = pd.read_csv(
            path,
            sep='::',
            engine='python',
            names=['userId', 'movieId', 'rating', 'timestamp'],
        )
    else:
        df = pd.read_csv(path)
    return df


def build_user_sequences(df, min_user_interactions=5):
    """
    Build chronological movie sequences for each user.
    """
    df = df[['userId', 'movieId', 'timestamp']].copy()
    df = df.sort_values(['userId', 'timestamp'])

    user_sequences = df.groupby('userId')['movieId'].apply(list).to_dict()

    filtered_sequences = {}
    for user_id, seq in user_sequences.items():
        if len(seq) >= min_user_interactions:
            filtered_sequences[user_id] = seq

    return filtered_sequences


def remap_movie_ids(user_sequences):
    """
    Remap raw movieIds into contiguous indices starting from 1.
    Index 0 is reserved for padding.
    """
    unique_items = sorted({item for seq in user_sequences.values() for item in seq})
    item2idx = {item: idx + 1 for idx, item in enumerate(unique_items)}
    idx2item = {idx: item for item, idx in item2idx.items()}

    remapped_sequences = {
        user_id: [item2idx[item] for item in seq]
        for user_id, seq in user_sequences.items()
    }

    return remapped_sequences, item2idx, idx2item


def split_user_sequences(user_sequences):
    """
    Leave-last-two-out chronological split.
    train = seq[:-2], val = seq[-2], test = seq[-1]
    """
    user_train, user_val, user_test = {}, {}, {}

    for user_id, seq in user_sequences.items():
        if len(seq) < 3:
            continue
        user_train[user_id] = seq[:-2]
        user_val[user_id] = seq[-2]
        user_test[user_id] = seq[-1]

    return user_train, user_val, user_test


df = load_ratings(CONFIG['ratings_path'])
print(df.head())
print('Raw interactions:', len(df))
print('Raw users:', df['userId'].nunique())
print('Raw movies:', df['movieId'].nunique())

user_sequences = build_user_sequences(df, min_user_interactions=CONFIG['min_user_interactions'])
remapped_sequences, item2idx, idx2item = remap_movie_ids(user_sequences)
user_train, user_val, user_test = split_user_sequences(remapped_sequences)

lengths = [len(seq) for seq in remapped_sequences.values()]
print('\nAfter filtering/remapping')
print('Users:', len(remapped_sequences))
print('Items:', len(item2idx))
print('Min sequence length:', min(lengths))
print('Mean sequence length:', np.mean(lengths))
print('Max sequence length:', max(lengths))
print('Train users:', len(user_train), 'Val users:', len(user_val), 'Test users:', len(user_test))

   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
Raw interactions: 1000209
Raw users: 6040
Raw movies: 3706

After filtering/remapping
Users: 6040
Items: 3706
Min sequence length: 20
Mean sequence length: 165.5975165562914
Max sequence length: 2314
Train users: 6040 Val users: 6040 Test users: 6040


## 5. SASRec dataset construction

In [ ]:
def pad_sequence(seq, max_len, pad_token=0):
    """
    Left-pad a sequence to max_len.
    """
    if len(seq) > max_len:
        seq = seq[-max_len:]
    if len(seq) < max_len:
        seq = [pad_token] * (max_len - len(seq)) + seq
    return seq


def build_sasrec_train_examples(user_train, max_len=50):
    examples = []
    for user_id, seq in user_train.items():
        for i in range(1, len(seq)):
            prefix_seq = seq[:i]
            target_item = seq[i]
            padded_prefix = pad_sequence(prefix_seq, max_len)
            examples.append((user_id, padded_prefix, target_item))
    return examples


def build_sasrec_eval_examples(user_train, user_val, user_test, max_len=50, split='val'):
    examples = []
    for user_id in user_train:
        train_seq = user_train[user_id]

        if split == 'val':
            input_seq = train_seq
            target_item = user_val[user_id]
        elif split == 'test':
            input_seq = train_seq + [user_val[user_id]]
            target_item = user_test[user_id]
        else:
            raise ValueError("split must be either 'val' or 'test'")

        padded_input = pad_sequence(input_seq, max_len)
        examples.append((user_id, padded_input, target_item))

    return examples


class SASRecTorchDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        user_id, input_seq, target_item = self.examples[idx]
        input_seq_tensor = torch.tensor(input_seq, dtype=torch.long)
        target_item_tensor = torch.tensor(target_item, dtype=torch.long)
        return input_seq_tensor, target_item_tensor


train_examples = build_sasrec_train_examples(user_train, max_len=CONFIG['max_len'])
val_examples = build_sasrec_eval_examples(user_train, user_val, user_test, max_len=CONFIG['max_len'], split='val')
test_examples = build_sasrec_eval_examples(user_train, user_val, user_test, max_len=CONFIG['max_len'], split='test')

train_dataset = SASRecTorchDataset(train_examples)
val_dataset = SASRecTorchDataset(val_examples)
test_dataset = SASRecTorchDataset(test_examples)

print('Train examples:', len(train_dataset))
print('Val examples:', len(val_dataset))
print('Test examples:', len(test_dataset))
print('Example input sequence shape:', train_dataset[0][0].shape)

Train examples: 982089
Val examples: 6040
Test examples: 6040
Example input sequence shape: torch.Size([20])


## 6. SASRec model

In [ ]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_len,
        hidden_dim=64,
        num_heads=2,
        num_layers=2,
        dropout=0.2,
    ):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(
            num_embeddings=num_items + 1,
            embedding_dim=hidden_dim,
            padding_idx=0,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_len,
            embedding_dim=hidden_dim,
        )

        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.output_bias = nn.Parameter(torch.zeros(num_items + 1))

    def _generate_causal_mask(self, seq_len, device):
        return torch.triu(
            torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
            diagonal=1,
        )

    def forward(self, input_seq):
        batch_size, seq_len = input_seq.size()
        device = input_seq.device

        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)

        item_emb = self.item_embedding(input_seq)
        pos_emb = self.position_embedding(positions)
        x = self.dropout(item_emb + pos_emb)

        padding_mask = input_seq.eq(0)
        causal_mask = self._generate_causal_mask(seq_len, device)

        hidden = self.encoder(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        hidden = self.layer_norm(hidden)

        # Important: sequences are left-padded, so the last position is the most recent valid item.
        seq_repr = hidden[:, -1, :]

        logits = seq_repr @ self.item_embedding.weight.T + self.output_bias
        logits[:, 0] = -1e9
        return logits


num_items = len(item2idx)
model = SASRec(
    num_items=num_items,
    max_len=CONFIG['max_len'],
    hidden_dim=CONFIG['hidden_dim'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
).to(DEVICE)

print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

SASRec(
  (item_embedding): Embedding(3707, 128, padding_idx=0)
  (position_embedding): Embedding(20, 128)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
)
Parameters: 877563


## 7. Training and multi-K evaluation functions

In [ ]:
def mask_seen_items(logits, input_seq):
    masked_logits = logits.clone()
    for i in range(input_seq.size(0)):
        seen_items = input_seq[i].unique()
        seen_items = seen_items[seen_items != 0]
        masked_logits[i, seen_items] = -1e9
    masked_logits[:, 0] = -1e9
    return masked_logits


def evaluate_model_multi_k(model, dataloader, device, topks=(10, 20)):
    model.eval()
    topks = sorted(set(topks))
    max_k = max(topks)

    metric_sums = {}
    for k in topks:
        metric_sums[f'Recall@{k}'] = 0.0
        metric_sums[f'NDCG@{k}'] = 0.0
    total_count = 0

    with torch.no_grad():
        for input_seq, target_item in dataloader:
            input_seq = input_seq.to(device, non_blocking=True)
            target_item = target_item.to(device, non_blocking=True)

            logits = model(input_seq)
            logits = mask_seen_items(logits, input_seq)
            _, topk_indices = torch.topk(logits, k=max_k, dim=1)

            targets = target_item.unsqueeze(1)
            for k in topks:
                topk_k = topk_indices[:, :k]
                hits = topk_k == targets
                recall = hits.any(dim=1).float()

                ndcg = torch.zeros_like(recall)
                for i in range(topk_k.size(0)):
                    hit_positions = torch.nonzero(hits[i], as_tuple=False)
                    if hit_positions.numel() > 0:
                        rank = hit_positions[0, 0].item() + 1
                        ndcg[i] = 1.0 / math.log2(rank + 1)

                metric_sums[f'Recall@{k}'] += recall.sum().item()
                metric_sums[f'NDCG@{k}'] += ndcg.sum().item()

            total_count += target_item.size(0)

    return {name: value / total_count for name, value in metric_sums.items()}


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for input_seq, target_item in dataloader:
        input_seq = input_seq.to(device, non_blocking=True)
        target_item = target_item.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(input_seq)
        loss = criterion(logits, target_item)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)


def format_metrics(metrics, topks):
    parts = []
    for k in topks:
        parts.append(f"Recall@{k}: {metrics[f'Recall@{k}']:.4f}")
        parts.append(f"NDCG@{k}: {metrics[f'NDCG@{k}']:.4f}")
    return ' | '.join(parts)

## 8. Build DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'),
)

print('Num train batches:', len(train_loader))
print('Num val batches:', len(val_loader))
print('Num test batches:', len(test_loader))

Num train batches: 1919
Num val batches: 12
Num test batches: 12


## 9. Train SASRec

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)
criterion = nn.CrossEntropyLoss()

best_val_recall10 = -1.0
history = []

print('Device:', DEVICE)
print('Users:', len(remapped_sequences))
print('Items:', num_items)
print('Train examples:', len(train_dataset))
print('Val examples:', len(val_dataset))
print('Test examples:', len(test_dataset))
print('Top-K metrics:', CONFIG['topks'])
print()

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_metrics = evaluate_model_multi_k(model, val_loader, DEVICE, topks=CONFIG['topks'])

    row = {'epoch': epoch, 'train_loss': train_loss, **val_metrics}
    history.append(row)

    print(
        f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | "
        f"Val {format_metrics(val_metrics, CONFIG['topks'])}"
    )

    if val_metrics['Recall@10'] > best_val_recall10:
        best_val_recall10 = val_metrics['Recall@10']
        torch.save(model.state_dict(), CONFIG['save_path'])

print('\nLoading best model for test evaluation...')
model.load_state_dict(torch.load(CONFIG['save_path'], map_location=DEVICE))
test_metrics = evaluate_model_multi_k(model, test_loader, DEVICE, topks=CONFIG['topks'])
print('Test', format_metrics(test_metrics, CONFIG['topks']))
print('Best model saved to:', CONFIG['save_path'])

history_df = pd.DataFrame(history)
history_df.tail()

Device: cuda
Users: 6040
Items: 3706
Train examples: 982089
Val examples: 6040
Test examples: 6040
Top-K metrics: [10, 20]

Epoch 01 | Train Loss: 9.2705 | Val Recall@10: 0.0197 | NDCG@10: 0.0089 | Recall@20: 0.0399 | NDCG@20: 0.0139
Epoch 02 | Train Loss: 7.4310 | Val Recall@10: 0.0637 | NDCG@10: 0.0321 | Recall@20: 0.1055 | NDCG@20: 0.0426
Epoch 03 | Train Loss: 6.8117 | Val Recall@10: 0.1291 | NDCG@10: 0.0695 | Recall@20: 0.2005 | NDCG@20: 0.0873
Epoch 04 | Train Loss: 6.3639 | Val Recall@10: 0.1760 | NDCG@10: 0.0957 | Recall@20: 0.2584 | NDCG@20: 0.1165
Epoch 05 | Train Loss: 6.1316 | Val Recall@10: 0.2070 | NDCG@10: 0.1113 | Recall@20: 0.2983 | NDCG@20: 0.1344
Epoch 06 | Train Loss: 5.9832 | Val Recall@10: 0.2237 | NDCG@10: 0.1213 | Recall@20: 0.3189 | NDCG@20: 0.1453
Epoch 07 | Train Loss: 5.8736 | Val Recall@10: 0.2358 | NDCG@10: 0.1279 | Recall@20: 0.3356 | NDCG@20: 0.1531
Epoch 08 | Train Loss: 5.7952 | Val Recall@10: 0.2475 | NDCG@10: 0.1354 | Recall@20: 0.3492 | NDCG@20: 0.1

,epoch,train_loss,Recall@10,NDCG@10,Recall@20,NDCG@20
25,26,5.407869,0.304470,0.173132,0.408278,0.199222
26,27,5.402899,0.305132,0.172901,0.403311,0.197751
27,28,5.399367,0.306623,0.171702,0.412086,0.198366
28,29,5.394966,0.306291,0.171416,0.409768,0.197514
29,30,5.390759,0.300497,0.169998,0.403974,0.196062


## 10. Save metrics and checkpoint

In [ ]:
RESULT_DIR = Path('/content/sasrec_ml1m_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

history_path = RESULT_DIR / 'training_history.csv'
test_path = RESULT_DIR / 'test_metrics.csv'

history_df.to_csv(history_path, index=False)
pd.DataFrame([test_metrics]).to_csv(test_path, index=False)

print('Saved history:', history_path)
print('Saved test metrics:', test_path)
print('Checkpoint:', CONFIG['save_path'])
!ls -lh /content/sasrec_ml1m_results

Saved history: /content/sasrec_ml1m_results/training_history.csv
Saved test metrics: /content/sasrec_ml1m_results/test_metrics.csv
Checkpoint: /content/sasrec_movielens1m_best.pth
total 8.0K
-rw-r--r-- 1 root root  116 May 10 19:31 test_metrics.csv
-rw-r--r-- 1 root root 3.0K May 10 19:31 training_history.csv


## 11. Optional: copy outputs to Google Drive

Uncomment this if you want to keep results after Colab disconnects.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sasrec_ml1m_results
# !cp -r /content/sasrec_ml1m_results/* /content/drive/MyDrive/sasrec_ml1m_results/
# !cp /content/sasrec_movielens1m_best.pth /content/drive/MyDrive/sasrec_ml1m_results/

# Appendix: local repo changes

To make your local repo read MovieLens1M `ratings.dat`, update `src/data/build_sequences.py`:

```python
def load_ratings(path):
    if path.endswith('.dat'):
        df = pd.read_csv(
            path,
            sep='::',
            engine='python',
            names=['userId', 'movieId', 'rating', 'timestamp'],
        )
    else:
        df = pd.read_csv(path)
    return df
```

Then download MovieLens1M locally and run:

```bash
python -m src.training.train_sasrec \
  --ratings_path data/raw/ml-1m/ml-1m/ratings.dat \
  --epochs 30 \
  --batch_size 512 \
  --max_len 20 \
  --hidden_dim 128 \
  --num_heads 4 \
  --dropout 0.3 \
  --topks 10,20
```
